# Dev 2: скоринг, охват аэропорта и гравитация

Актуальная витрина Dev 1 из `main` после обновления `7eff240`.
Хеши входов закреплены в `src/scoring/reference/current_data.json`.
Режим `auto` выбирает **предварительный exploratory**: доля пересадок через
Москву отсутствует, туризм задан прокси. Гравитация — отдельный сценарный индекс.

Методики: [скоринг](../src/scoring/README.md),
[охват и гравитация](../src/scoring/GRAVITY_MODEL.md).


In [1]:
from pathlib import Path
import sys
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/scoring/city_scorer.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.schemas.contracts import ScoringWeights
from src.scoring.city_scorer import (
    load_features, score_cities, weight_sensitivity, get_scored_cities,
    calculate_effective_population, calculate_gravity_matrix,
)

DATA_PATH = ROOT / "data/processed/cities_features.parquet"
ROUTES_PATH = ROOT / "data/processed/routes_proof.parquet"
manifest = json.loads((ROOT / "src/scoring/reference/current_data.json").read_text(encoding="utf-8"))
for rel, item in manifest["files"].items():
    assert hashlib.sha256((ROOT / rel).read_bytes()).hexdigest() == item["sha256"], f"Вход {rel} изменился; обновите отчёты и manifest"
features, routes = load_features(DATA_PATH), load_features(ROUTES_PATH)
print("Репозиторий:", manifest["repository"], "main:", manifest["base_main_commit"])
print("Городов:", len(features), "направлений:", len(routes))
display(features[["iata", "city_name", "population_mln", "airport_pax_mln",
                  "dist_to_mow_km", "competitor_gap_score", "tourism_demand_score"]])

Репозиторий: https://github.com/Prepodavately/RedWings__hackathon main: 5a526c0a27a2f10aee85b1e80ef57e2f5c6c5ef0
Городов: 12 направлений: 132


,iata,city_name,population_mln,airport_pax_mln,dist_to_mow_km,competitor_gap_score,tourism_demand_score
0,CEK,Челябинск,1.474039,2.167,1497,66.0,34.57
1,GOJ,Нижний Новгород,1.813433,1.400,387,54.8,28.08
2,GSV,Саратов,1.245017,1.064,719,45.0,6.70
3,KUF,Самара,2.096344,3.600,844,83.2,28.74
4,KZN,Казань,1.706065,5.400,730,79.4,63.00
5,MRV,Минеральные Воды,0.648282,4.600,1339,60.1,50.00
6,OMS,Омск,1.201893,1.630,2233,75.8,20.10
7,PEE,Пермь,1.270293,2.005,1141,75.0,39.17
8,SVX,Екатеринбург,1.911571,8.385,1430,66.7,70.63
9,TJM,Тюмень,1.018211,2.760,1699,53.7,65.33


## Линейный скоринг

Население нормируется min-max по полному лонглисту в 0–100. Равные значения получают 50.
Расстояние до Москвы оценивается функцией `distance_sweet_spot_score`:

`D = max(35, 100 - 65 * abs(dist_to_mow_km - 1150) / 1200)`.

Сам балл расстояния округляется до двух знаков по дополнению к ТЗ;
взвешенная сумма и ранжирование далее не округляются. Это существующее правило `aviationModel.ts` с кусочно-линейным максимумом на 1150 км.
Оно является сценарной эвристикой команды; технический оптимум SJ-100 не доказан.
Расстояние до SVX можно использовать как дополнительный признак анализа, но оно
больше не участвует в этом балле. Два индекса Dev 1 уже заданы в 0–100.

Веса exploratory: 30/20/30/20. В расширенном case добавляется доля московских
пересадок с весом 20%, остальные веса становятся 24/16/24/16. Это экспертные
приоритеты, а не обученные на исторических исходах коэффициенты.

Для нового хаба SVX исключается из топ-3. Общий API по умолчанию рассматривает
p2p и сохраняет все 12 DTO для карты/селектора. При явном hub исключённые строки
тоже остаются в DTO, но получают is_top3=False и позицию в конце списка.


In [2]:
import warnings

weights = ScoringWeights(0.30, 0.20, 0.30, 0.20)
with warnings.catch_warnings(record=True) as notices:
    warnings.simplefilter("always")
    hub = score_cities(features, weights, hypothesis="hub")
    p2p = score_cities(features, weights, hypothesis="p2p")
    dtos = get_scored_cities(weights, path=DATA_PATH)
print("Автовыбор режима:", hub.mode, "— ПРЕДВАРИТЕЛЬНО")
print("Зафиксировано предупреждений:", len(notices))
print("DTO для интерфейса:", len(dtos), "городов; SVX:", any(c.iata == "SVX" for c in dtos))
assert len(dtos) == 12 and any(c.iata == "SVX" for c in dtos)
assert sum(c.is_top3 for c in dtos) == 3
print("Эффективные веса:", hub.weights)
for name, result in [("Новый хаб", hub), ("Города прямой сети", p2p)]:
    print(name, "— ПРЕДВАРИТЕЛЬНО")
    display(result.top3[["rank", "iata", "city_name", "total_score", "result_status"]])

Автовыбор режима: exploratory — ПРЕДВАРИТЕЛЬНО
Зафиксировано предупреждений: 3
DTO для интерфейса: 12 городов; SVX: True
Эффективные веса: {'population': 0.3, 'hub_distance': 0.2, 'competitor_gap': 0.3, 'tourism_demand': 0.2}
Новый хаб — ПРЕДВАРИТЕЛЬНО


,rank,iata,city_name,total_score,result_status
0,1,KUF,Самара,77.392000,preliminary
1,2,KZN,Казань,73.784455,preliminary
2,3,UFA,Уфа,64.540409,preliminary


Города прямой сети — ПРЕДВАРИТЕЛЬНО


,rank,iata,city_name,total_score,result_status
0,1,KUF,Самара,77.392000,preliminary
1,2,SVX,Екатеринбург,77.273994,preliminary
2,3,KZN,Казань,73.784455,preliminary


## Из чего складывается балл

Каждый сегмент — вклад критерия, сумма сегментов равна итоговому баллу.
Туризм — индекс зарплаты/авиаподвижности. Обновлённый показатель конкуренции
Dev 1 — обратная доля главного перевозчика среди всех вылетов аэропорта за
10–15.11.2026, а не доля свободного рынка. Исходные источники сохранены в витрине.


In [3]:
import plotly.graph_objects as go

eligible = hub.table.loc[hub.table.eligible].sort_values("total_score")
labels = {"population": "Население", "hub_distance": "Расстояние до Москвы",
          "competitor_gap": "Ниша по индексу Dev 1", "tourism_demand": "Прокси туризма"}
fig = go.Figure()
for criterion, label in labels.items():
    fig.add_bar(y=eligible.city_name, x=eligible[f"contribution_{criterion}"],
                name=label, orientation="h")
fig.update_layout(barmode="stack", title="Новый хаб: предварительный скоринг четырёх критериев",
                  xaxis_title="Вклад в итог, баллы / 100", yaxis_title="",
                  template="plotly_white", height=520, legend_orientation="h")
fig.show(renderer="plotly_mimetype")

## Устойчивость к весам

2 000 случайных наборов: каждый базовый вес × Uniform(0,8; 1,2), затем нормировка;
seed=42. Данные и границы шкал фиксированы. Частота топ-3 — характеристика этого
эксперимента, не вероятность успеха. Нулевой вес в таком анализе остаётся нулевым.


In [4]:
hub_stability = weight_sensitivity(hub, n_samples=2000, relative_change=0.20, seed=42)
p2p_stability = weight_sensitivity(p2p, n_samples=2000, relative_change=0.20, seed=42)
display(hub_stability)
display(p2p_stability.head(4))
assert hub_stability.top3_frequency.sum() == 3.0
assert p2p_stability.top3_frequency.sum() == 3.0
print("Хаб: состав топ-3 постоянен, порядок также постоянен при этих вариациях.")
print("P2P: состав топ-3 постоянен, Самара и Екатеринбург могут меняться местами.")

,iata,city_name,rank,result_status,top3_frequency,best_rank,worst_rank,mean_rank,n_samples
0,KUF,Самара,1,preliminary,1.0,1,1,1.0000,2000
1,KZN,Казань,2,preliminary,1.0,2,2,2.0000,2000
2,UFA,Уфа,3,preliminary,1.0,3,3,3.0000,2000
3,PEE,Пермь,4,preliminary,0.0,4,4,4.0000,2000
4,CEK,Челябинск,5,preliminary,0.0,5,6,5.0190,2000
5,GOJ,Нижний Новгород,6,preliminary,0.0,5,6,5.9810,2000
6,VOG,Волгоград,7,preliminary,0.0,7,8,7.2845,2000
7,TJM,Тюмень,8,preliminary,0.0,7,8,7.7155,2000
8,OMS,Омск,9,preliminary,0.0,9,10,9.4080,2000
9,MRV,Минеральные Воды,10,preliminary,0.0,9,11,9.6185,2000


,iata,city_name,rank,result_status,top3_frequency,best_rank,worst_rank,mean_rank,n_samples
0,KUF,Самара,1,preliminary,1.0,1,2,1.457,2000
1,SVX,Екатеринбург,2,preliminary,1.0,1,2,1.543,2000
2,KZN,Казань,3,preliminary,1.0,3,3,3.000,2000
3,UFA,Уфа,4,preliminary,0.0,4,4,4.000,2000


Хаб: состав топ-3 постоянен, порядок также постоянен при этих вариациях.
P2P: состав топ-3 постоянен, Самара и Екатеринбург могут меняться местами.


## Фича А: эффективное население

`P_eff = P_city + Σ P_sat · exp(−λt)` при `t ≤ 2 ч`, λ=0,45 1/час.
Ядро берётся целиком; население спутников в скобках — люди, результат — млн.
Времена доезда заданы сценарным словарём из ТЗ, по умолчанию 0,65 ч.
Это оценка охвата, **не измеренная дорожная изохрона**. Слагаемые ниже позволяют
проверить любой вклад. Никакие неописанные спутники не добавляются.


In [5]:
LAM, GAMMA = 0.45, 1.15
cities = features.copy(deep=True)
effective = calculate_effective_population(cities, lam=LAM)
cities["effective_population_mln"] = effective
catchment = pd.DataFrame(effective.attrs["catchment_details"])
member_sum = catchment.groupby("iata").satellite_population.sum()
cities["population_from_members_mln"] = cities.population_city_mln + cities.iata.map(member_sum).fillna(0) / 1e6
cities["population_members_gap_mln"] = cities.population_mln - cities.population_from_members_mln
display(cities[["iata", "city_name", "population_mln", "effective_population_mln"]].sort_values("effective_population_mln", ascending=False))
display(catchment.loc[catchment.iata.isin(["KUF", "MRV"])])
print("Расхождения суммы ядра и спутников с полной агломерацией:")
display(cities.loc[cities.population_members_gap_mln.abs() > 1e-6,
    ["iata", "population_mln", "population_from_members_mln", "population_members_gap_mln"]])
assert (cities.effective_population_mln >= cities.population_city_mln).all()
assert (cities.effective_population_mln <= cities.population_from_members_mln).all()

,iata,city_name,population_mln,effective_population_mln
3,KUF,Самара,2.096344,1.835678
8,SVX,Екатеринбург,1.911571,1.830651
1,GOJ,Нижний Новгород,1.813433,1.671159
4,KZN,Казань,1.706065,1.610649
0,CEK,Челябинск,1.474039,1.398650
10,UFA,Уфа,1.309873,1.279537
11,VOG,Волгоград,1.325253,1.251183
7,PEE,Пермь,1.270293,1.208725
6,OMS,Омск,1.201893,1.176399
2,GSV,Саратов,1.245017,1.162890


,iata,satellite,satellite_population,access_time_h,time_source,within_two_hours,decay_weight,effective_contribution_mln
9,KUF,Городской округ Тольятти,662683,0.75,scenario_override,True,0.713552,0.472859
10,KUF,Муниципальный район Волжский,132763,0.65,scenario_default,True,0.746395,0.099094
11,KUF,Муниципальный район Ставропольский,90149,0.65,scenario_default,True,0.746395,0.067287
12,KUF,Муниципальный район Красноярский,56427,0.65,scenario_default,True,0.746395,0.042117
17,MRV,Городской округ Город-курорт Пятигорск,211304,0.35,scenario_override,True,0.854277,0.180512
18,MRV,Городской округ Город-курорт Кисловодск,132977,0.75,scenario_override,True,0.713552,0.094886
19,MRV,Городской округ Город-курорт Ессентуки,124038,0.55,scenario_override,True,0.780750,0.096843
20,MRV,Городской округ Город-курорт Железноводск,49818,0.65,scenario_default,True,0.746395,0.037184


Расхождения суммы ядра и спутников с полной агломерацией:


,iata,population_mln,population_from_members_mln,population_members_gap_mln
5,MRV,0.648282,0.586898,0.061384


У MRV сумма перечисленных компонентов меньше полной агломерации на
**61 384 человека**. Эффективная масса посчитана только по описанным компонентам;
Dev 1 нужно уточнить исходный состав.

## Фича Б: гравитационная матрица

`G_ij = P_eff_i · P_eff_j · sqrt(q_i q_j) · W(d) / [d^γ · (1 + F_ij)]`, γ=1,15.

- `q` — зарплата региона / средняя зарплата кандидатов.
- `d` — гаверсинус между аэропортами, км, радиус Земли 6371 км.
- `W=0` до 300 и от 3000 км, `W=1` на 800–1500 км; переходы линейные.
- `F_ij` — направленная частота конкурентов из `routes_proof.parquet`.
  При отсутствии/NaN берётся 0 и сохраняется признак отсутствия наблюдения.

Пороги, λ/γ и времена явно согласованы как допущения. Индекс относительный,
не пассажиропоток и не сертификационная характеристика самолёта.


In [6]:
gravity = calculate_gravity_matrix(cities, routes, gamma=GAMMA)
cities["gravity_hub_potential"] = cities.iata.map(gravity.sum(axis=1))
cities["purchasing_power_index"] = cities.iata.map(gravity.attrs["purchasing_power_index"])
comparison = cities[["iata", "city_name", "effective_population_mln", "gravity_hub_potential"]].merge(
    hub.table[["iata", "rank", "total_score", "eligible"]], on="iata", validate="one_to_one"
).sort_values(["gravity_hub_potential", "iata"], ascending=[False, True])
comparison["gravity_rank_all"] = range(1, len(comparison) + 1)
print("Ненаблюдаемых частот с подстановкой нуля:", gravity.attrs["missing_competition_pairs"])
print("Все 12 городов; rank — место допустимого кандидата в линейном скоринге хаба:")
display(comparison)
print("Гравитационный топ-3 допустимых кандидатов нового хаба:")
display(comparison.loc[comparison.eligible].head(3))
assert gravity.shape == (12, 12)
assert (np.diag(gravity) == 0).all()
assert np.isfinite(gravity.to_numpy()).all()
np.testing.assert_allclose(cities.gravity_hub_potential, cities.iata.map(gravity.attrs["gravity_hub_potential"]))

Ненаблюдаемых частот с подстановкой нуля: 0
Все 12 городов; rank — место допустимого кандидата в линейном скоринге хаба:


,iata,city_name,effective_population_mln,gravity_hub_potential,rank,total_score,eligible,gravity_rank_all
1,GOJ,Нижний Новгород,1.671159,0.005081,6,57.928835,True,1
3,KUF,Самара,1.835678,0.004976,1,77.392000,True,2
0,CEK,Челябинск,1.398650,0.004960,5,60.061493,True,3
11,VOG,Волгоград,1.251183,0.004770,7,51.797042,True,4
6,OMS,Омск,1.176399,0.004200,9,46.497350,True,5
7,PEE,Пермь,1.208725,0.004079,4,63.122416,True,6
2,GSV,Саратов,1.162890,0.003668,11,42.532765,True,7
4,KZN,Казань,1.610649,0.003402,2,73.784455,True,8
10,UFA,Уфа,1.279537,0.002994,3,64.540409,True,9
9,TJM,Тюмень,0.981151,0.002875,8,50.891947,True,10


Гравитационный топ-3 допустимых кандидатов нового хаба:


,iata,city_name,effective_population_mln,gravity_hub_potential,rank,total_score,eligible,gravity_rank_all
1,GOJ,Нижний Новгород,1.671159,0.005081,6,57.928835,True,1
3,KUF,Самара,1.835678,0.004976,1,77.392000,True,2
0,CEK,Челябинск,1.398650,0.004960,5,60.061493,True,3


## Две модели дают разные рейтинги

Линейный топ-3 хаба: **Самара, Казань, Уфа**. Гравитационный топ-3:
**Нижний Новгород, Самара, Челябинск**. Казань занимает 8-е место из 12.
Согласованная формула не подтверждает совместное лидерство Самары и Казани;
коэффициенты под ожидаемый вывод не менялись.

Цвет ниже показывает вклад отдельного направления в сумму по строке.
SVX остаётся пунктом назначения, хотя не является кандидатом нового хаба.
Частоты направленные, поэтому матрица может быть несимметрична.


In [7]:
fig_gravity = go.Figure(go.Heatmap(
    z=gravity.to_numpy() * 1000,
    x=gravity.columns.tolist(), y=gravity.index.tolist(),
    colorscale="Blues", colorbar_title="G × 10³",
    hovertemplate="%{y} → %{x}<br>G × 10³ = %{z:.4f}<extra></extra>",
))
fig_gravity.update_layout(title="Потенциал связей: сценарная гравитация, не пассажиры",
    xaxis_title="Назначение", yaxis_title="Отправление", template="plotly_white", height=620)
fig_gravity.update_yaxes(autorange="reversed")
fig_gravity.show(renderer="plotly_mimetype")

## Пробелы в данных

Автовыбор не добавляет выдуманную долю пересадок, а явно использует четыре
доступных критерия. Явный case отклоняет эту же витрину. Ошибка ниже ожидаема.
Аудит каждой пары город/критерий сохраняет отсутствующие метаданные видимыми.


In [8]:
try:
    score_cities(features, weights, mode="case")
except ValueError as error:
    print("Ожидаемый блокер полного режима:", error)
else:
    raise AssertionError("Снимок без данных о пересадках не должен проходить полный режим")

audit = hub.audit
missing = audit.assign(missing_source=audit.src.isna() | audit.src.fillna("").astype(str).str.strip().eq(""))
display(missing.groupby("criterion").agg(rows=("iata", "size"), missing_sources=("missing_source", "sum")))
for limitation in hub.limitations:
    print("•", limitation)

Ожидаемый блокер полного режима: Отсутствует колонка moscow_transfer_share_pct


,rows,missing_sources
criterion,,
airport_feasibility,12,12
airport_pax,12,0
competitor_gap,12,12
hub_distance,12,12
moscow_transfer,12,12
population,12,0
tourism_demand,12,12


• Веса — экспертное предположение; результат условен относительно данных и критериев.
• Рейтинг городов не доказывает спрос, прибыльность маршрутов или победу гипотезы.
• Источники записаны поставщиком данных; наличие ссылки не означает внешнюю проверку.
• ПРЕДВАРИТЕЛЬНО: доля пассажиров с пересадкой в Москве не участвует в формуле.
• Происхождение, сопоставимость периодов и возможности аэропортов требуют проверки.
• tourism_demand_score и competitor_gap_score используются в определении Dev 1; проверьте прокси.
• Предварительный рейтинг: auto выбрал exploratory; нет полей для case: airport_feasible, method_airport_feasibility, method_airport_pax, method_competitor_gap, method_hub_distance, method_moscow_transfer, method_population, method_tourism_demand, moscow_transfer_share_pct, period_airport_feasibility, period_airport_pax, period_competitor_gap, period_hub_distance, period_moscow_transfer, period_population, period_tourism_demand, src_airport_feasibility, src_competitor_gap, src_h

## Передача команде

**Dev 1:** уточнить расхождение населения MRV, O&D-долю московских пересадок,
сопоставимый показатель туризма, периоды и возможности аэропортов.

**Dev 4:** базовый `get_scored_cities(weights)` сохраняет все 12 DTO; при отсутствии
стандартного Parquet используется загрузчик Dev 1 из локальных CSV.
Предварительный статус нужно подписать в UI. Для новой аналитики доступны
`calculate_effective_population` и `calculate_gravity_matrix`; замороженный DTO не меняется.

**Капитан:** гравитация — проверяемая гипотеза об относительном потенциале связей.
Для прогноза пассажиров нужна калибровка на O&D-данных и расчёт экономики А→Б.
Текущая модель не подтверждает совместное лидерство Самары и Казани.

CSV всех направлений, слагаемые охвата, ранги, λ/γ и хеши входов:
`src/scoring/reports/hub/`, `src/scoring/reports/p2p/`.
Команды воспроизведения — `src/scoring/README.md`, журнал ИИ — `src/scoring/AI_USAGE.md`.
